<div class="alert alert-block alert-info">

### Imports & Connection
</div>

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import datetime as datetime
import plotly.graph_objects as go

from scipy import stats
from pyhive import presto
from IPython.display import display
from datetime import datetime, timedelta
# from scipy.stats import gaussian_kde

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
print('Notebook dir - ', os.getcwd())
local_extract = '/Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark'
print('Data dir - ', local_extract)

Notebook dir -  /Users/E2074/analytics/customer/Ads-Monetisation/monet-benchmark
Data dir -  /Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark


<div class="alert alert-block alert-info">
    
## Datasets & Parameter
</div>

In [5]:
## Parameter 
start_date = '20241128'
end_date = '20241204'

city_list = [  'Bangalore', 'Hyderabad', 'Delhi', 'Chennai', 'Ahmedabad',
                'Kolkata', 'Lucknow', 'Jaipur', 'Pune', 'Chandigarh', 
                'Vijayawada', 'Mumbai', 'Indore', 'Tirupati', 'Patna', 
                 'Coimbatore', 'Guwahati', 'Vishakapatnam']

<div class="alert alert-block alert-warning">
    
#### Required Query Function 
</div>

In [6]:
def func_ad_response(date,segment_date, city):
    
    
    base_query = f"""


    with response AS (
        
            SELECT
                res.yyyymmdd as yyyymmdd,
                res.city as city,
                res.os as os,
                res.responsetype as responsetype,  
                res.userid as userid,
                res.slotname as slotname, 
                res.format as format, 
                res.responsestatus as responsestatus,
                -- res.isprefetched as isprefetched,
                coalesce(res.aduuid, res.adid) as response_id,
                REPLACE(REGEXP_EXTRACT(metadata1, 'title=([^,]*)'), 'title=', '') AS title,
                REPLACE(REGEXP_EXTRACT(metadata1, 'advertiserName=([^,]*)'), 'advertiserName=', '') AS advertiserName,
                REPLACE(REGEXP_EXTRACT(metadata1, 'body=([^,]*)'), 'body=', '') AS body
            FROM 
                canonical.iceberg_log_telemetry_ads_impressions_immutable res
            WHERE 
                res.yyyymmdd = '{date}'
                AND res.eventName = 'Ad_Response'
                AND res.responsetype IN ('GAMBanner', 'ONLINE_SALES', 'System Banner')
                AND res.city = '{city}'
                -- AND NOT regexp_like(res.format, 'Arrived|OTW|Started|nativeBanner')
                AND (res.responsestatus = 'true' OR res.responsestatus IS NULL)

                -- AND quarter_hour = '1115'
        ),

        segment as (

        SELECT
            customer_id,
            taxi_ltr_segment,
            taxi_retention_segments taxi_retention_segment,
            taxi_regularity_segment taxi_rr_regularity_segment,
            -- taxi_intent_segment taxi_rr_intent_segment,
            taxi_fe_regularity_segment
            -- taxi_fe_intent_segment
        FROM 
            datasets.iallocator_customer_segments
        WHERE 
            run_date = '{segment_date}'
            AND taxi_recency_segment != 'INACTIVE'
        )
        
        SELECT
            response.yyyymmdd as yyyymmdd,
            response.city as city,
            response.os as os,
            response.responsetype as responsetype,  
            response.userid as userid,
            response.slotname as slotname, 
            response.format as format, 
            response.responsestatus as responsestatus,
            -- response.isprefetched as isprefetched,
            
            taxi_ltr_segment,
            taxi_retention_segment,
            taxi_rr_regularity_segment,
            -- taxi_rr_intent_segment,
            taxi_fe_regularity_segment,
            -- taxi_fe_intent_segment,

            response_id,
            
            title,
            advertiserName,
            body
            
        FROM    
            response
        LEFT JOIN 
            segment as segment
            ON response.userid = segment.customer_id

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base



In [7]:
def func_ad_impression(date, city):
    
    
    base_query = f"""
    
        SELECT
            DISTINCT
            coalesce(imp.aduuid, imp.adid) as impression_id,
            imp.userid as userid,
            imp.yyyymmdd as yyyymmdd
        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable as imp
        WHERE 
            imp.yyyymmdd = '{date}'
            AND imp.eventName = 'Ad_Impression'
            AND imp.responsetype IN ('GAMBanner', 'ONLINE_SALES', 'System Banner')
            AND imp.city = '{city}'
            -- AND quarter_hour = '1115'

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [8]:
def func_ad_viewable_impression(date, city):
    
    
    base_query = f"""
    
        SELECT
            DISTINCT
            coalesce(view_imp.aduuid, view_imp.adid) as viewable_impression_id,
            view_imp.userid as userid,
            view_imp.yyyymmdd as yyyymmdd
        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable view_imp
        WHERE 
            view_imp.yyyymmdd = '{date}'
            AND view_imp.eventName = 'Ad_Viewable_Impression'
            AND view_imp.responsetype IN ('GAMBanner', 'ONLINE_SALES', 'System Banner')
            AND view_imp.city = '{city}'
            -- AND quarter_hour = '1115'

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [9]:
def func_ad_click(date, city):
    
    
    base_query = f"""

            
        SELECT
            DISTINCT
            coalesce(clk.aduuid, clk.adid) as click_id,
            clk.userid as userid,
            clk.yyyymmdd as yyyymmdd
        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable as clk
        WHERE 
            clk.yyyymmdd = '{date}'
            AND clk.eventName = 'Ad_Click'
            AND clk.responsetype IN ('GAMBanner', 'ONLINE_SALES', 'System Banner')
            AND clk.city = '{city}'
            -- AND quarter_hour = '1115'

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [10]:
def func_pull_ads_data(start_date, end_date, city_list):
    
    date_list = pd.date_range(start=start_date, end=end_date).strftime('%Y%m%d').tolist()
    formatted_end_date = datetime.strptime(end_date, '%Y%m%d').strftime('%Y-%m-%d')
    
    for date in date_list:

        for city in city_list:
            
            print(date, city)
            df_response_data_dump = func_ad_response(date, formatted_end_date, city)
            print('response')
            # display(df_response_data_dump.head(5))
            
            df_impression_data_dump = func_ad_impression(date, city)
            print('impression')
            # display(df_impression_data_dump.head(5))
            
            df_viewable_impression_data_dump = func_ad_viewable_impression(date, city)
            print('viewability')
            # display(df_viewable_impression_data_dump.head(5))
            
            df_click_data_dump = func_ad_click(date, city)
            print('click')
            # display(df_click_data_dump.head(5))
            
            merge1 = pd.merge(df_response_data_dump, df_impression_data_dump, 
                              how='left', 
                              left_on=['yyyymmdd', 'userid', 'response_id'],
                              right_on=['yyyymmdd', 'userid', 'impression_id'])
            print('merge 1')
            merge2 = pd.merge(merge1, df_viewable_impression_data_dump, 
                              how='left', 
                              left_on=['yyyymmdd', 'userid', 'impression_id'],
                              right_on=['yyyymmdd', 'userid', 'viewable_impression_id'])
            print('merge 2')
            merge3 = pd.merge(merge2, df_click_data_dump, 
                              how='left', 
                              left_on=['yyyymmdd', 'userid', 'response_id'],
                              right_on=['yyyymmdd', 'userid', 'click_id'])
            print('merge 3')
            merge3.to_parquet(local_extract + '/data_dump/{}_{}_data.parquet'.format(date,city), index=False)

    # folder_path = local_extract + '/data_dump'
    # parquet_files = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith('.parquet')]
    # df = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)
    
    return merge3

In [11]:
# df = func_pull_ads_data(start_date, end_date, city_list)

<div class="alert alert-block alert-warning">
    
#### User Defined Function & Required Variable 
</div>

In [12]:
pattern_format = r"OTW|Arrived|Started|nativeBanner|default|MAP_BRANDING"
pattern_os = r"iPadOS"

agg_cols = {
    'responses' : ('response_id', 'nunique'),
    'impressions' : ('impression_id', 'nunique'),
    'viewable_impressions' : ('viewable_impression_id', 'nunique'),
    'clicks' : ('click_id', 'nunique'),
}

In [13]:
def func_percentage(df):
    df['render_rate'] = (df['impressions']*100.00/df['responses']).round(2)
    df['view_rate'] = (df['viewable_impressions']*100.00/df['impressions']).round(2)
    df['ctr'] = (df['clicks']*100.00/df['viewable_impressions']).round(2)
    return df

def group_aggregate_data(df, groupby_cols, agg_col):
    df_grouped = df.groupby(groupby_cols).agg(**agg_col).reset_index()
    df_grouped = func_percentage(df_grouped)
    df_grouped.replace([np.inf, -np.inf], 0.0, inplace=True)
    df_grouped.fillna(0.0, inplace=True)
    
    return df_grouped

<div class="alert alert-block alert-warning">
    
#### Reading Extracted data
</div>

In [14]:
prefix_list = pd.date_range(start=start_date, end=end_date).strftime('%Y%m%d').tolist()
cities_list = ['_Bangalore_' ] # '_Bangalore_', '_Chennai_', '_Delhi_', '_Hyderabad_'

path_to_directory = local_extract + '/data_dump/'
dataframes = []

In [15]:
for prefix in prefix_list[0:-2]:
   
    print(f"Processing date: {prefix}")
    parquet_files = glob.glob(os.path.join(path_to_directory, f"{prefix}*.parquet"))
    filtered_cities = [file for file in parquet_files if any(city in file for city in cities_list)]

    for city in filtered_cities:
        dataframes.append(pd.read_parquet(city))
        print(f"Loaded city: {city}")
        

df_data = pd.concat(dataframes, ignore_index=True)
df_data = df_data[~df_data["format"].str.contains(pattern_format, regex=True)]
df_data = df_data[~df_data["os"].str.contains(pattern_os, regex=True)]

Processing date: 20241128
Loaded city: /Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark/data_dump/20241128_Bangalore_data.parquet
Processing date: 20241129
Loaded city: /Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark/data_dump/20241129_Bangalore_data.parquet
Processing date: 20241130
Loaded city: /Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark/data_dump/20241130_Bangalore_data.parquet
Processing date: 20241201
Loaded city: /Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark/data_dump/20241201_Bangalore_data.parquet
Processing date: 20241202
Loaded city: /Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark/data_dump/20241202_Bangalore_data.parquet


In [16]:
df_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20911882 entries, 0 to 25769331
Data columns (total 19 columns):
 #   Column                      Dtype 
---  ------                      ----- 
 0   yyyymmdd                    object
 1   city                        object
 2   os                          object
 3   responsetype                object
 4   userid                      object
 5   slotname                    object
 6   format                      object
 7   responsestatus              object
 8   taxi_ltr_segment            object
 9   taxi_retention_segment      object
 10  taxi_rr_regularity_segment  object
 11  taxi_fe_regularity_segment  object
 12  response_id                 object
 13  title                       object
 14  advertiserName              object
 15  body                        object
 16  impression_id               object
 17  viewable_impression_id      object
 18  click_id                    object
dtypes: object(19)
memory usage: 3.1+ GB


In [17]:
df_data.shape

(20911882, 19)

In [18]:
df_data.head(1)

,yyyymmdd,city,os,responsetype,userid,slotname,format,responsestatus,taxi_ltr_segment,taxi_retention_segment,taxi_rr_regularity_segment,taxi_fe_regularity_segment,response_id,title,advertiserName,body,impression_id,viewable_impression_id,click_id
0,20241128,Bangalore,iOS,GAMBanner,657552fac8fa733b90051452,CaptainSearch1,gamCombinedBanner,true,PHH,PLATINUM,WEEKLY,WEEKLY,657552fac8fa733b90051452_1A9C28C2-B454-4FF4-B1...,None,None,None,None,None,None


<div class="alert alert-block alert-success">  
    
## Analysis
</div>

In [19]:
df_data.columns

Index(['yyyymmdd', 'city', 'os', 'responsetype', 'userid', 'slotname',
       'format', 'responsestatus', 'taxi_ltr_segment',
       'taxi_retention_segment', 'taxi_rr_regularity_segment',
       'taxi_fe_regularity_segment', 'response_id', 'title', 'advertiserName',
       'body', 'impression_id', 'viewable_impression_id', 'click_id'],
      dtype='object')

In [20]:
df_data.head(3)

,yyyymmdd,city,os,responsetype,userid,slotname,format,responsestatus,taxi_ltr_segment,taxi_retention_segment,taxi_rr_regularity_segment,taxi_fe_regularity_segment,response_id,title,advertiserName,body,impression_id,viewable_impression_id,click_id
0,20241128,Bangalore,iOS,GAMBanner,657552fac8fa733b90051452,CaptainSearch1,gamCombinedBanner,true,PHH,PLATINUM,WEEKLY,WEEKLY,657552fac8fa733b90051452_1A9C28C2-B454-4FF4-B1...,None,None,None,None,None,None
1,20241128,Bangalore,iOS,GAMBanner,616d8da04c6ba12e89d8b525,CaptainSearch1,gamCombinedBanner,true,PHH,PLATINUM,WEEKLY,BI_WEEKLY,616d8da04c6ba12e89d8b525_C454B261-193C-49EC-95...,None,None,None,616d8da04c6ba12e89d8b525_C454B261-193C-49EC-95...,616d8da04c6ba12e89d8b525_C454B261-193C-49EC-95...,None
2,20241128,Bangalore,iOS,GAMBanner,623507fca63f41b8e3d6295c,PostOrderArrived4,gamCombinedBanner,true,PHH,PLATINUM,WEEKLY,WEEKLY,623507fca63f41b8e3d6295c_25F21B28-1005-4325-82...,None,None,None,None,None,None


In [21]:
df = df_data[~df_data['responsetype'].isin(['ONLINE_SALES'])]

<div class="alert alert-block alert-success">  
    
#### Overall Numbers
</div>

In [22]:
group_aggregate_data(df, 
                     ['city'], 
                     agg_cols)

,city,responses,impressions,viewable_impressions,clicks,render_rate,view_rate,ctr
0,Bangalore,19456564,6228062,5128649,43107,32.01,82.35,0.84


In [23]:
group_aggregate_data(df, 
                     ['yyyymmdd'], 
                     agg_cols)

,yyyymmdd,responses,impressions,viewable_impressions,clicks,render_rate,view_rate,ctr
0,20241128,3576553,1137709,938957,6162,31.81,82.53,0.66
1,20241129,3701932,1203815,995039,6712,32.52,82.66,0.67
2,20241130,3948877,1338127,1104887,7887,33.89,82.57,0.71
3,20241201,3796374,1211432,992539,9797,31.91,81.93,0.99
4,20241202,4451078,1342757,1101409,12549,30.17,82.03,1.14


In [31]:
group_aggregate_data(df, 
                     ['os'], 
                     agg_cols)

,os,responses,impressions,viewable_impressions,clicks,render_rate,view_rate,ctr
0,android,17092425,5064013,4120085,28965,29.63,81.36,0.7
1,iOS,2364139,1164049,1008564,14142,49.24,86.64,1.4


In [24]:
group_aggregate_data(df, 
                     ['yyyymmdd', 'os'], 
                     agg_cols)

,yyyymmdd,os,responses,impressions,viewable_impressions,clicks,render_rate,view_rate,ctr
0,20241128,android,3168067,934361,762388,4135,29.49,81.59,0.54
1,20241128,iOS,408486,203348,176569,2027,49.78,86.83,1.15
2,20241129,android,3234892,971953,794543,4261,30.05,81.75,0.54
3,20241129,iOS,467040,231862,200496,2451,49.64,86.47,1.22
4,20241130,android,3463465,1092411,893234,4738,31.54,81.77,0.53
5,20241130,iOS,485412,245716,211653,3149,50.62,86.14,1.49
6,20241201,android,3329996,983191,794756,6745,29.53,80.83,0.85
7,20241201,iOS,466378,228241,197783,3052,48.94,86.66,1.54
8,20241202,android,3914033,1087875,879346,9086,27.79,80.83,1.03
9,20241202,iOS,537045,254882,222063,3463,47.46,87.12,1.56


In [25]:
group_aggregate_data(df, 
                     ['os', 'responsetype'], 
                     agg_cols)

,os,responsetype,responses,impressions,viewable_impressions,clicks,render_rate,view_rate,ctr
0,android,GAMBanner,15741422,3965498,3256151,22072,25.19,82.11,0.68
1,android,System Banner,1368666,1111594,872327,6959,81.22,78.48,0.80
2,iOS,GAMBanner,2364139,1164049,1008564,14142,49.24,86.64,1.40


In [26]:
group_aggregate_data(df, 
                     ['os', 'responsetype', 'format'], 
                     agg_cols)

,os,responsetype,format,responses,impressions,viewable_impressions,clicks,render_rate,view_rate,ctr
0,android,GAMBanner,,142303,86397,70880,862,60.71,82.04,1.22
1,android,GAMBanner,gamCombinedBanner,15561286,3879129,3185291,21212,24.93,82.11,0.67
2,android,GAMBanner,nativeVideoBanner,38451,131,125,3,0.34,95.42,2.40
3,android,System Banner,ONLINE_SALES,1780,1659,1587,18,93.20,95.66,1.13
4,android,System Banner,Rewards,1366887,1109936,870741,6941,81.20,78.45,0.80
5,iOS,GAMBanner,gamCombinedBanner,2364139,1164049,1008564,14142,49.24,86.64,1.40


In [27]:
group_aggregate_data(df, 
                     ['os', 'responsetype', 'taxi_ltr_segment'], 
                     agg_cols)

,os,responsetype,taxi_ltr_segment,responses,impressions,viewable_impressions,clicks,render_rate,view_rate,ctr
0,android,GAMBanner,HH,1175861,262782,225804,2293,22.35,85.93,1.02
1,android,GAMBanner,PHH,14337843,3663488,2998076,19205,25.55,81.84,0.64
2,android,GAMBanner,UNKNOWN,99731,15353,12470,248,15.39,81.22,1.99
3,android,System Banner,HH,62024,52186,42612,470,84.14,81.65,1.10
4,android,System Banner,PHH,1303246,1056574,827548,6465,81.07,78.32,0.78
5,android,System Banner,UNKNOWN,120,103,48,0,85.83,46.60,0.00
6,iOS,GAMBanner,HH,68633,34935,29921,448,50.90,85.65,1.50
7,iOS,GAMBanner,PHH,2280986,1123221,973509,13598,49.24,86.67,1.40
8,iOS,GAMBanner,UNKNOWN,3333,1361,1188,14,40.83,87.29,1.18


In [28]:
group_aggregate_data(df, 
                     ['os', 'responsetype', 'taxi_retention_segment'], 
                     agg_cols)

,os,responsetype,taxi_retention_segment,responses,impressions,viewable_impressions,clicks,render_rate,view_rate,ctr
0,android,GAMBanner,DORMANT,290972,64467,52695,808,22.16,81.74,1.53
1,android,GAMBanner,ELITE,5421552,1410575,1131629,6070,26.02,80.22,0.54
2,android,GAMBanner,GOLD,3021006,756075,630578,4575,25.03,83.40,0.73
3,android,GAMBanner,HH,1119951,251479,216543,2099,22.45,86.11,0.97
4,android,GAMBanner,PLATINUM,4016573,1032120,844797,5302,25.70,81.85,0.63
5,android,GAMBanner,PRIME,162260,41883,33651,241,25.81,80.35,0.72
6,android,GAMBanner,SILVER,1481390,369671,313987,2403,24.95,84.94,0.77
7,android,GAMBanner,UNKNOWN,99731,15353,12470,248,15.39,81.22,1.99
8,android,System Banner,DORMANT,16215,13408,10156,119,82.69,75.75,1.17
9,android,System Banner,ELITE,483057,388368,297675,1735,80.40,76.65,0.58


In [29]:
group_aggregate_data(df, 
                     ['os', 'responsetype', 'taxi_rr_regularity_segment'], 
                     agg_cols)

,os,responsetype,taxi_rr_regularity_segment,responses,impressions,viewable_impressions,clicks,render_rate,view_rate,ctr
0,android,GAMBanner,BI_WEEKLY,3077271,774366,642635,4654,25.16,82.99,0.72
1,android,GAMBanner,DAILY,4506976,1188772,954487,5209,26.38,80.29,0.55
2,android,GAMBanner,MONTHLY,1237255,299722,251736,1975,24.22,83.99,0.78
3,android,GAMBanner,RARE_NEED,736962,127370,107513,814,17.28,84.41,0.76
4,android,GAMBanner,UNKNOWN,1275592,278135,238274,2541,21.80,85.67,1.07
5,android,GAMBanner,WEEKLY,4779379,1273258,1041705,6553,26.64,81.81,0.63
6,android,System Banner,BI_WEEKLY,292384,239001,188431,1658,81.74,78.84,0.88
7,android,System Banner,DAILY,384313,308874,237860,1354,80.37,77.01,0.57
8,android,System Banner,MONTHLY,114057,93402,75764,734,81.89,81.12,0.97
9,android,System Banner,RARE_NEED,92670,76492,60489,823,82.54,79.08,1.36


In [32]:
df.head(5)

,yyyymmdd,city,os,responsetype,userid,slotname,format,responsestatus,taxi_ltr_segment,taxi_retention_segment,taxi_rr_regularity_segment,taxi_fe_regularity_segment,response_id,title,advertiserName,body,impression_id,viewable_impression_id,click_id
0,20241128,Bangalore,iOS,GAMBanner,657552fac8fa733b90051452,CaptainSearch1,gamCombinedBanner,true,PHH,PLATINUM,WEEKLY,WEEKLY,657552fac8fa733b90051452_1A9C28C2-B454-4FF4-B1...,None,None,None,None,None,None
1,20241128,Bangalore,iOS,GAMBanner,616d8da04c6ba12e89d8b525,CaptainSearch1,gamCombinedBanner,true,PHH,PLATINUM,WEEKLY,BI_WEEKLY,616d8da04c6ba12e89d8b525_C454B261-193C-49EC-95...,None,None,None,616d8da04c6ba12e89d8b525_C454B261-193C-49EC-95...,616d8da04c6ba12e89d8b525_C454B261-193C-49EC-95...,None
2,20241128,Bangalore,iOS,GAMBanner,623507fca63f41b8e3d6295c,PostOrderArrived4,gamCombinedBanner,true,PHH,PLATINUM,WEEKLY,WEEKLY,623507fca63f41b8e3d6295c_25F21B28-1005-4325-82...,None,None,None,None,None,None
3,20241128,Bangalore,iOS,GAMBanner,63b3abfad32f0f81302f78b0,PostOrderArrived3,gamCombinedBanner,true,PHH,SILVER,BI_WEEKLY,BI_WEEKLY,63b3abfad32f0f81302f78b0_FB374D99-247A-44FE-95...,None,None,None,63b3abfad32f0f81302f78b0_FB374D99-247A-44FE-95...,63b3abfad32f0f81302f78b0_FB374D99-247A-44FE-95...,None
4,20241128,Bangalore,iOS,GAMBanner,63b3abfad32f0f81302f78b0,CaptainSearch1,gamCombinedBanner,true,PHH,SILVER,BI_WEEKLY,BI_WEEKLY,63b3abfad32f0f81302f78b0_A1499222-A46D-406B-9C...,None,None,None,63b3abfad32f0f81302f78b0_A1499222-A46D-406B-9C...,63b3abfad32f0f81302f78b0_A1499222-A46D-406B-9C...,None


In [22]:
# source (GAM/OS/SB)
# os (AND/iOS)
# format (Image/Video/Story)

In [23]:
# taxi_ltr_segment (HH/PHH)
# taxi_retention_segment (SGPE)
# taxi_rr_regularity_segment (Daily/Weekly/Monthly)

In [24]:
# slotname(slot1/2/3)

In [25]:
# metadata-category